# Four-threshold directional-change linkage

This notebook links the three adjacent causal building blocks for GBPUSD.r:

1. small 0.20% → medium 0.40%;
2. medium 0.40% → large 0.60%;
3. large 0.60% → macro 0.80%.

A valid lineage begins when a small DCC leaves broad four-scale agreement. The original small event remains the immutable lineage ID. Exact medium and large event IDs are hand-off keys; they do not replace the seed.

Terminal depth is hindsight until the lineage resolves:

    depth 0: stopped below medium
    depth 1: reached medium, stopped below large
    depth 2: reached large, stopped below macro
    depth 3: reached macro

No live checkpoint features, HMM, classifier, or trading rule is introduced.

In [1]:
from enum import StrEnum
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots

from dc_basics.data import resolve_data_root
from dc_basics.core import dc_live_features, parse_dc_events

DATA_ROOT = resolve_data_root()  # $DC_DATA_ROOT or .dc_data_root
SYMBOL = "GBPUSD.r"
DATA_PATH = DATA_ROOT / f"{SYMBOL}.parquet"
SMALL_THETA = 0.002
MEDIUM_THETA = 0.004
LARGE_THETA = 0.006
MACRO_THETA = 0.008
CASEBOOK_BARS_BEFORE = 150
CASEBOOK_BARS_AFTER = 150

if not 0.0 < SMALL_THETA < MEDIUM_THETA < LARGE_THETA < MACRO_THETA:
    raise ValueError("Thresholds must satisfy 0 < small < medium < large < macro.")

print(f"Symbol: {SYMBOL}")
print(
    f"Thresholds: {SMALL_THETA:.2%}, {MEDIUM_THETA:.2%}, "
    f"{LARGE_THETA:.2%}, {MACRO_THETA:.2%}"
)

Symbol: GBPUSD.r
Thresholds: 0.20%, 0.40%, 0.60%, 0.80%


In [2]:
class Scale(StrEnum):
    SMALL = "small"
    MEDIUM = "medium"
    LARGE = "large"
    MACRO = "macro"


from dc_basics.enums import DcMode


from dc_basics.enums import AttemptOutcome


class TerminalOutcome(StrEnum):
    STOPPED_BELOW_MEDIUM = "stopped_below_medium"
    STOPPED_BELOW_LARGE = "stopped_below_large"
    STOPPED_BELOW_MACRO = "stopped_below_macro"
    REACHED_MACRO = "reached_macro"
    AMBIGUOUS_SMALL_MEDIUM = "ambiguous_small_medium"
    AMBIGUOUS_MEDIUM_LARGE = "ambiguous_medium_large"
    AMBIGUOUS_LARGE_MACRO = "ambiguous_large_macro"
    UNRESOLVED_BELOW_MEDIUM = "unresolved_below_medium"
    UNRESOLVED_BELOW_LARGE = "unresolved_below_large"
    UNRESOLVED_BELOW_MACRO = "unresolved_below_macro"
    UNLINKED_AFTER_MEDIUM = "unlinked_after_medium"
    UNLINKED_AFTER_LARGE = "unlinked_after_large"


SCALE_ORDER = (Scale.SMALL, Scale.MEDIUM, Scale.LARGE, Scale.MACRO)
SCALE_SETTINGS = {
    Scale.SMALL: {"prefix": "S", "theta": SMALL_THETA, "color": "#16a34a"},
    Scale.MEDIUM: {"prefix": "M", "theta": MEDIUM_THETA, "color": "#2563eb"},
    Scale.LARGE: {"prefix": "L", "theta": LARGE_THETA, "color": "#7c3aed"},
    Scale.MACRO: {"prefix": "X", "theta": MACRO_THETA, "color": "#c026d3"},
}
MODE_MAP = {1.0: DcMode.UP.value, -1.0: DcMode.DOWN.value}

## Genealogical and causal contract

The previous bar must have all four modes aligned. The initiating small DCC establishes the target direction.

A stage propagates only through the exact upper-scale DCC that resolves the lower contest. That same event ID must initiate the next adjacent contest. Direction, timestamp, and prior upper mode must agree across the hand-off.

Same-bar propagation (the lower DCC and the upper DCC that joins it on one bar) is contemporaneously known, so it contains no unresolved live interval. It is still a resolved propagation. Ambiguous, unresolved, and unlinked paths have no terminal depth: they remain explicit in the audits and are excluded from every probability ladder. The strict-clean cohort additionally drops lineages with a same-bar stage; it is needed for live-interval work, but its reach rates are biased low (see below).

In [3]:
raw = pd.read_parquet(DATA_PATH)
if not isinstance(raw.index, pd.DatetimeIndex):
    if "time" not in raw.columns:
        raise TypeError("Expected a DatetimeIndex or a 'time' column.")
    raw = raw.set_index(pd.to_datetime(raw.pop("time")))
raw = raw.sort_index()

if raw.index.has_duplicates:
    raise ValueError(f"Found {raw.index.duplicated().sum():,} duplicate timestamps.")
if "close" not in raw.columns:
    raise KeyError("The input data has no close column.")
prices = raw["close"].astype(np.float64).rename(SYMBOL)
if not np.isfinite(prices.to_numpy()).all() or prices.le(0.0).any():
    raise ValueError("Close prices must be finite and strictly positive.")

print(f"Bars: {len(prices):,}")
print(f"Range: {prices.index[0]} to {prices.index[-1]}")

Bars: 258,573
Range: 2023-05-22 14:00:00+02:00 to 2026-11-09 13:30:00+02:00


In [4]:
from dc_basics.events import prepare_event_ledger


event_ledgers = {}
live_frames = {}
mode_frame = pd.DataFrame(index=prices.index)
for scale in SCALE_ORDER:
    settings = SCALE_SETTINGS[scale]
    raw_events = parse_dc_events(prices=prices, theta=settings["theta"])
    ledger = prepare_event_ledger(raw_events, settings["prefix"], scale, settings["theta"])
    live = dc_live_features(prices, theta=settings["theta"])
    event_ledgers[scale] = ledger
    live_frames[scale] = live
    mode_frame[f"{scale.value}_mode"] = live["direction"].map(MODE_MAP)
    mode_frame[f"{scale.value}_upward_dcc"] = live["upward_dcc"].astype(bool)
    mode_frame[f"{scale.value}_downward_dcc"] = live["downward_dcc"].astype(bool)
    event_lookup = ledger.set_index("dcc_idx")["event_id"]
    mode_frame[f"{scale.value}_event_id"] = mode_frame.index.to_series().map(event_lookup)

clock_summary = pd.DataFrame.from_records([
    {
        "scale": scale.value,
        "threshold": SCALE_SETTINGS[scale]["theta"],
        "events": len(event_ledgers[scale]),
        "first_mode_time": live_frames[scale]["direction"].first_valid_index(),
    }
    for scale in SCALE_ORDER
]).set_index("scale")
clock_summary.style.format({"threshold": "{:.2%}", "events": "{:,}"})

,threshold,events,first_mode_time
scale,,,
small,0.20%,"2,697",2023-05-22 16:15:00+02:00
medium,0.40%,839,2023-05-22 17:50:00+02:00
large,0.60%,392,2023-05-23 11:30:00+02:00
macro,0.80%,220,2023-05-24 13:35:00+02:00


## Reconstruct the three adjacent contests

In [5]:
from dc_basics.hierarchy import build_adjacent_attempts


In [6]:
small_medium_attempts = build_adjacent_attempts(mode_frame, Scale.SMALL, Scale.MEDIUM)
medium_large_attempts = build_adjacent_attempts(mode_frame, Scale.MEDIUM, Scale.LARGE)
large_macro_attempts = build_adjacent_attempts(mode_frame, Scale.LARGE, Scale.MACRO)
adjacent_attempts = pd.concat(
    [small_medium_attempts, medium_large_attempts, large_macro_attempts],
    ignore_index=True,
)
adjacent_attempts["outcome_information_time"] = adjacent_attempts["resolution_time"]
adjacent_attempts["outcome_is_hindsight_at_start"] = adjacent_attempts[
    "outcome_information_time"
].gt(adjacent_attempts["start_time"])

adjacent_audit = adjacent_attempts.groupby(["edge", "outcome"], sort=False).agg(
    attempts=("initiating_event_id", "size"),
    median_resolution_bars=("resolution_bars", "median"),
)
adjacent_audit.style.format({
    "attempts": "{:,}",
    "median_resolution_bars": "{:.1f}",
})

D:\projects\quant-studies\signalfoundry\.venv\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


## Pin broad-agreement small seeds

A lineage root is eligible only when the previous bar has all four modes equal. This gives the small disturbance a definite common-scale ancestry.

In [7]:
previous_modes = mode_frame[
    ["small_mode", "medium_mode", "large_mode", "macro_mode"]
].shift(1)
seed_attempts = small_medium_attempts.copy()
seed_times = pd.DatetimeIndex(seed_attempts["start_time"])
for scale in SCALE_ORDER:
    mode_column = f"{scale.value}_mode"
    seed_attempts[f"prior_{scale.value}_mode"] = previous_modes.loc[
        seed_times, mode_column
    ].to_numpy()
seed_attempts["broad_agreement_before_seed"] = (
    seed_attempts["prior_small_mode"].eq(seed_attempts["prior_medium_mode"])
    & seed_attempts["prior_medium_mode"].eq(seed_attempts["prior_large_mode"])
    & seed_attempts["prior_large_mode"].eq(seed_attempts["prior_macro_mode"])
)
broad_seeds = seed_attempts.loc[seed_attempts["broad_agreement_before_seed"]].copy()

print(f"All small-to-medium attempts: {len(seed_attempts):,}")
print(f"Broad four-scale origins: {len(broad_seeds):,}")

All small-to-medium attempts: 1,768
Broad four-scale origins: 1,006


## Link three stages by exact event IDs

In [8]:
from dc_basics.lineage import build_four_scale_linkages


linkages = build_four_scale_linkages(
    broad_seeds, medium_large_attempts, large_macro_attempts
)

## Integrity, exclusions, and exact hand-offs

Resolved rows (reached macro, or stopped by a rejection) receive depths 0–3. Strict-clean lineages additionally require every traversed stage to contain at least one unresolved bar, which excludes every same-bar hand-off.

A same-bar stage can only be a propagation, because a rejection needs a *later* lower DCC. Strict-clean filtering therefore removes propagations only, and every strict-clean conditional reach rate is biased low relative to the all-resolved cohort.

In [9]:
resolved_linkages = linkages.loc[linkages["terminal_depth"].notna()].copy()

stage_one_resolution = resolved_linkages["terminal_position"].where(
    resolved_linkages["terminal_depth"].eq(0), resolved_linkages["medium_position"]
)
stage_two_resolution = resolved_linkages["terminal_position"].where(
    resolved_linkages["terminal_depth"].eq(1), resolved_linkages["large_position"]
)
stage_one_has_live_bar = pd.to_numeric(stage_one_resolution).gt(
    resolved_linkages["seed_position"]
)
stage_two_has_live_bar = (
    resolved_linkages["terminal_depth"].eq(0)
    | pd.to_numeric(stage_two_resolution).gt(
        pd.to_numeric(resolved_linkages["medium_position"])
    )
)
stage_three_has_live_bar = (
    resolved_linkages["terminal_depth"].le(1)
    | pd.to_numeric(resolved_linkages["terminal_position"]).gt(
        pd.to_numeric(resolved_linkages["large_position"])
    )
)

strict_clean_linkages = resolved_linkages.loc[
    resolved_linkages["terminal_label_hindsight_at_seed"]
    & ~resolved_linkages["small_medium_same_bar"]
    & ~resolved_linkages["medium_large_same_bar"]
    & ~resolved_linkages["large_macro_same_bar"]
    & stage_one_has_live_bar
    & stage_two_has_live_bar
    & stage_three_has_live_bar
].copy()
if strict_clean_linkages.empty:
    raise ValueError("No strict-clean four-scale lineages were found.")
if not resolved_linkages.index.equals(linkages.index[linkages["resolved_lineage"]]):
    raise AssertionError("The local resolved cohort disagrees with the library's resolved_lineage flag.")
if not strict_clean_linkages.index.equals(linkages.index[linkages["strict_clean"]]):
    raise AssertionError("The local strict-clean cohort disagrees with the library's strict_clean flag.")

strict_depth_one_plus = strict_clean_linkages["terminal_depth"].ge(1)
strict_depth_two_plus = strict_clean_linkages["terminal_depth"].ge(2)
if not strict_clean_linkages.loc[
    strict_depth_one_plus, "medium_position"
].gt(strict_clean_linkages.loc[strict_depth_one_plus, "seed_position"]).all():
    raise AssertionError("A strict-clean Stage-1 lineage has no live bar.")
if not strict_clean_linkages.loc[
    strict_depth_two_plus, "large_position"
].gt(strict_clean_linkages.loc[strict_depth_two_plus, "medium_position"]).all():
    raise AssertionError("A strict-clean Stage-2 lineage has no live bar.")
if not strict_clean_linkages.loc[
    strict_depth_two_plus, "terminal_position"
].gt(strict_clean_linkages.loc[strict_depth_two_plus, "large_position"]).all():
    raise AssertionError("A strict-clean Stage-3 lineage has no live bar.")

terminal_outcome_audit = linkages.groupby("terminal_outcome", dropna=False, sort=False).agg(
    lineages=("seed_small_event_id", "size"),
    median_total_resolution_bars=("total_resolution_bars", "median"),
)
exclusion_audit = pd.Series({
    "broad_agreement_seeds": len(linkages),
    "resolved_lineages": len(resolved_linkages),
    "small_medium_same_bar": int(resolved_linkages["small_medium_same_bar"].sum()),
    "medium_large_same_bar": int(resolved_linkages["medium_large_same_bar"].sum()),
    "large_macro_same_bar": int(resolved_linkages["large_macro_same_bar"].sum()),
    "strict_clean_lineages": len(strict_clean_linkages),
})
display(terminal_outcome_audit.style.format({
    "lineages": "{:,}",
    "median_total_resolution_bars": "{:.1f}",
}))
display(exclusion_audit)

,lineages,median_total_resolution_bars
terminal_outcome,,
stopped_below_medium,531,42.0
reached_macro,220,215.5
stopped_below_macro,85,342.0
stopped_below_large,169,198.0
unresolved_below_large,1,nan


broad_agreement_seeds         1006
resolved_monotone_lineages    1005
small_medium_same_bar           26
medium_large_same_bar           13
large_macro_same_bar             5
strict_clean_lineages          970
dtype: int64

In [10]:
stage_two_by_event = medium_large_attempts.set_index("initiating_event_id", drop=False)
stage_three_by_event = large_macro_attempts.set_index("initiating_event_id", drop=False)
medium_handoffs = linkages.loc[linkages["medium_event_id"].notna()].copy()
large_handoffs = linkages.loc[linkages["large_event_id"].notna()].copy()

if not medium_handoffs["medium_event_id"].isin(stage_two_by_event.index).all():
    raise AssertionError("A medium hand-off lacks its exact second-stage contest.")
if not large_handoffs["large_event_id"].isin(stage_three_by_event.index).all():
    raise AssertionError("A large hand-off lacks its exact third-stage contest.")

medium_stage_rows = stage_two_by_event.loc[medium_handoffs["medium_event_id"]]
large_stage_rows = stage_three_by_event.loc[large_handoffs["large_event_id"]]
if not medium_handoffs["medium_time"].reset_index(drop=True).eq(
    medium_stage_rows["start_time"].reset_index(drop=True)
).all():
    raise AssertionError("A medium hand-off timestamp is inconsistent.")
if not large_handoffs["large_time"].reset_index(drop=True).eq(
    large_stage_rows["start_time"].reset_index(drop=True)
).all():
    raise AssertionError("A large hand-off timestamp is inconsistent.")

handoff_audit = pd.Series({
    "small_medium_propagations_from_broad_seeds": int(
        broad_seeds["outcome"].eq(AttemptOutcome.PROPAGATED.value).sum()
    ),
    "exact_medium_handoffs": len(medium_handoffs),
    "medium_large_propagations_in_lineages": int(
        linkages["medium_large_outcome"].eq(AttemptOutcome.PROPAGATED.value).sum()
    ),
    "exact_large_handoffs": len(large_handoffs),
    "unlinked_after_medium": int(
        linkages["terminal_outcome"].eq(TerminalOutcome.UNLINKED_AFTER_MEDIUM.value).sum()
    ),
    "unlinked_after_large": int(
        linkages["terminal_outcome"].eq(TerminalOutcome.UNLINKED_AFTER_LARGE.value).sum()
    ),
})
handoff_audit

small_medium_propagations_from_broad_seeds    475
exact_medium_handoffs                         475
medium_large_propagations_in_lineages         305
exact_large_handoffs                          305
unlinked_after_medium                           0
unlinked_after_large                            0
dtype: int64

## Linked conditional-probability ladder

All probabilities within a row use one internally consistent cohort. The product identity is arithmetic conditioning, not an independence assumption:

    P(reach macro from seed)
      = P(reach medium)
      × P(reach large | medium)
      × P(reach macro | large)

Because reaching macro implies reaching large implies reaching medium, the identity holds by arithmetic in any cohort; it is not a test. The checks that *can* fail are genealogical: a lineage counted at a depth must carry the exact hand-off events and propagated edge outcomes that depth requires.

The all-resolved cohort is unbiased. The strict-clean cohort drops same-bar propagations only, so its rates are lower by construction. These are full-sample hindsight frequencies, not calibrated forecasts.

In [11]:
def summarize_linked_probabilities(sample: pd.DataFrame, sample_name: str) -> dict[str, object]:
    reached_medium = sample["terminal_depth"].ge(1)
    reached_large = sample["terminal_depth"].ge(2)
    reached_macro = sample["terminal_depth"].eq(3)
    # Genealogical consistency: these checks can fail if the linkage were broken.
    required = {
        "small_medium_outcome": reached_medium,
        "medium_large_outcome": reached_large,
        "large_macro_outcome": reached_macro,
    }
    for column, mask in required.items():
        if not sample.loc[mask, column].eq(AttemptOutcome.PROPAGATED.value).all():
            raise AssertionError(f"A lineage counted at its depth lacks a propagated {column}.")
    if not sample.loc[reached_medium, "medium_event_id"].notna().all():
        raise AssertionError("A lineage counted as reaching medium has no exact medium hand-off.")
    if not sample.loc[reached_large, "large_event_id"].notna().all():
        raise AssertionError("A lineage counted as reaching large has no exact large hand-off.")
    if sample.loc[~reached_large, "large_event_id"].notna().any():
        raise AssertionError("A lineage below depth 2 carries a large hand-off.")
    return {
        "sample": sample_name,
        "lineages": len(sample),
        "p_reach_medium": reached_medium.mean(),
        "p_reach_large_given_medium": reached_large.sum() / reached_medium.sum(),
        "p_reach_macro_given_large": reached_macro.sum() / reached_large.sum(),
        "p_reach_macro_from_seed": reached_macro.mean(),
    }


probability_summary = pd.DataFrame.from_records([
    summarize_linked_probabilities(resolved_linkages, "all resolved (unbiased)"),
    summarize_linked_probabilities(strict_clean_linkages, "strict clean (no same-bar stage; biased low)"),
]).set_index("sample")
probability_summary.style.format({
    "lineages": "{:,}",
    "p_reach_medium": "{:.2%}",
    "p_reach_large_given_medium": "{:.2%}",
    "p_reach_macro_given_large": "{:.2%}",
    "p_reach_macro_from_seed": "{:.2%}",
})

,lineages,p_reach_medium,p_reach_large_given_medium,p_reach_macro_given_large,p_reach_macro_from_seed,product_check
sample,,,,,,
all resolved monotone,"1,005",47.16%,64.35%,72.13%,21.89%,21.89%
strict clean future,970,45.26%,62.41%,71.17%,20.10%,20.10%


In [12]:
depth_counts = strict_clean_linkages["terminal_depth"].value_counts().sort_index()
depth_flow_figure = go.Figure(go.Sankey(
    arrangement="snap",
    node={
        "label": [
            "Broad-agreement small seeds",
            "Stopped below medium",
            "Reached medium",
            "Stopped below large",
            "Reached large",
            "Stopped below macro",
            "Reached macro",
        ],
        "color": [
            "#111827", "#64748b", "#2563eb", "#64748b",
            "#7c3aed", "#64748b", "#c026d3",
        ],
        "pad": 20,
        "thickness": 18,
    },
    link={
        "source": [0, 0, 2, 2, 4, 4],
        "target": [1, 2, 3, 4, 5, 6],
        "value": [
            int(depth_counts.get(0, 0)),
            int(depth_counts.get(1, 0) + depth_counts.get(2, 0) + depth_counts.get(3, 0)),
            int(depth_counts.get(1, 0)),
            int(depth_counts.get(2, 0) + depth_counts.get(3, 0)),
            int(depth_counts.get(2, 0)),
            int(depth_counts.get(3, 0)),
        ],
        "color": [
            "rgba(100,116,139,0.45)", "rgba(37,99,235,0.40)",
            "rgba(100,116,139,0.45)", "rgba(124,58,237,0.40)",
            "rgba(100,116,139,0.45)", "rgba(192,38,211,0.45)",
        ],
    },
))
depth_flow_figure.update_layout(
    title=f"{SYMBOL}: strict-clean four-scale propagation depth",
    template="plotly_white",
    height=600,
)
depth_flow_figure.show()

## Stage properties and resolution anatomy

Stage duration begins at its hand-off DCC and ends at the next propagation or rejection DCC. We compare all adjacent contests with the exact contests inherited by broad-origin lineages.

In [13]:
from dc_basics.audit import summarize_stage


linked_medium_ids = set(linkages.loc[linkages["medium_event_id"].notna(), "medium_event_id"])
linked_large_ids = set(linkages.loc[linkages["large_event_id"].notna(), "large_event_id"])
linked_stage_two = medium_large_attempts.loc[
    medium_large_attempts["initiating_event_id"].isin(linked_medium_ids)
]
linked_stage_three = large_macro_attempts.loc[
    large_macro_attempts["initiating_event_id"].isin(linked_large_ids)
]

stage_property_summary = pd.DataFrame.from_records([
    summarize_stage(small_medium_attempts, "all small->medium"),
    summarize_stage(broad_seeds, "broad-origin small->medium"),
    summarize_stage(medium_large_attempts, "all medium->large"),
    summarize_stage(linked_stage_two, "linked medium->large"),
    summarize_stage(large_macro_attempts, "all large->macro"),
    summarize_stage(linked_stage_three, "linked large->macro"),
]).set_index("stage")
stage_property_summary.style.format({
    "attempts": "{:,}",
    "resolved_attempts": "{:,}",
    "propagation_rate": "{:.2%}",
    "median_resolution_bars": "{:.1f}",
    "same_bar_propagations": "{:,}",
    "same_bar_rate": "{:.2%}",
    "clean_future_attempts": "{:,}",
    "clean_propagation_rate": "{:.2%}",
    "median_clean_resolution_bars": "{:.1f}",
})

,attempts,resolved_attempts,propagation_rate,median_resolution_bars,same_bar_rate
stage,,,,,
all small->medium,1768,"1,768",0.474548,34.000000,0.024887
broad-origin small->medium,1006,"1,006",0.472167,37.000000,0.025845
all medium->large,616,615,0.637398,69.000000,0.024390
linked medium->large,475,474,0.643460,73.000000,0.027426
all large->macro,305,305,0.721311,113.000000,0.016393
linked large->macro,305,305,0.721311,113.000000,0.016393


### Reading the stage table

- `propagation_rate` counts every resolved attempt, including zero-bar (same-bar) propagations. `clean_propagation_rate` uses only `clean_future_attempts`, the attempts with at least one unresolved bar, so it is lower by construction. They are different quantities.
- Here the "linked" rows differ from the "all" rows because a lineage root needs *four*-scale agreement: medium->large attempts whose seed left a three-scale agreement against an opposed macro are not linked. In `3_threshold_dc_linkage`, with three-scale roots, the two rows coincide.

The nested counts below are strictly *inside* each stage: lower-scale DCCs after the stage starts and before its resolving bar. A DCC on the resolution bar itself is not counted.

In [14]:
from dc_basics.audit import count_events_inside


small_positions = event_ledgers[Scale.SMALL]["dcc_pos"].to_numpy(dtype=np.int64)
medium_positions = event_ledgers[Scale.MEDIUM]["dcc_pos"].to_numpy(dtype=np.int64)
strict_clean_linkages["small_dccs_inside_stage_two"] = pd.NA
strict_clean_linkages["small_dccs_inside_stage_three"] = pd.NA
strict_clean_linkages["medium_dccs_inside_stage_three"] = pd.NA

for row_index, lineage in strict_clean_linkages.iterrows():
    depth = int(lineage["terminal_depth"])
    if depth >= 1:
        stage_two_end = (
            int(lineage["terminal_position"])
            if depth == 1
            else int(lineage["large_position"])
        )
        strict_clean_linkages.loc[row_index, "small_dccs_inside_stage_two"] = count_events_inside(
            small_positions, int(lineage["medium_position"]), stage_two_end, include_end=False
        )
    if depth >= 2:
        strict_clean_linkages.loc[row_index, "small_dccs_inside_stage_three"] = count_events_inside(
            small_positions, int(lineage["large_position"]), int(lineage["terminal_position"]),
            include_end=False,
        )
        strict_clean_linkages.loc[row_index, "medium_dccs_inside_stage_three"] = count_events_inside(
            medium_positions, int(lineage["large_position"]), int(lineage["terminal_position"]),
            include_end=False,
        )

for column in [
    "small_dccs_inside_stage_two",
    "small_dccs_inside_stage_three",
    "medium_dccs_inside_stage_three",
]:
    strict_clean_linkages[column] = strict_clean_linkages[column].astype("Int64")

duration_summary = strict_clean_linkages.groupby("terminal_depth", sort=True).agg(
    lineages=("seed_small_event_id", "size"),
    stage_one_bars_median=("small_medium_bars", "median"),
    stage_two_bars_median=("medium_large_bars", "median"),
    stage_three_bars_median=("large_macro_bars", "median"),
    total_bars_median=("total_resolution_bars", "median"),
    total_bars_p90=("total_resolution_bars", lambda values: values.quantile(0.90)),
    small_dccs_stage_two_median=("small_dccs_inside_stage_two", "median"),
    small_dccs_stage_three_median=("small_dccs_inside_stage_three", "median"),
    medium_dccs_stage_three_median=("medium_dccs_inside_stage_three", "median"),
)
duration_summary.style.format({
    "lineages": "{:,}",
    "stage_one_bars_median": "{:.1f}",
    "stage_two_bars_median": "{:.1f}",
    "stage_three_bars_median": "{:.1f}",
    "total_bars_median": "{:.1f}",
    "total_bars_p90": "{:.1f}",
    "small_dccs_stage_two_median": "{:.1f}",
    "small_dccs_stage_three_median": "{:.1f}",
    "medium_dccs_stage_three_median": "{:.1f}",
})

D:\projects\quant-studies\signalfoundry\.venv\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
D:\projects\quant-studies\signalfoundry\.venv\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
D:\projects\quant-studies\signalfoundry\.venv\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


,lineages,stage_one_bars_median,stage_two_bars_median,stage_three_bars_median,total_bars_median,total_bars_p90,small_dccs_stage_two_median,small_dccs_stage_three_median,medium_dccs_stage_three_median
terminal_depth,,,,,,,,,
0,531,42.0,nan,nan,42.0,165.0,,,
1,165,33.0,142.0,nan,200.0,494.8,1.0,,
2,79,34.0,50.0,226.0,352.0,725.0,0.0,3.0,1.0
3,195,31.0,57.0,65.0,236.0,657.2,0.0,0.0,0.0


## Directional and chronological stability

Terminal depth remains a hindsight diagnostic. These strata check whether the linked flow is dominated by one direction or a small part of the sample. They use the all-resolved cohort, the unbiased one; the strict-clean filter would remove same-bar propagations only.

In [15]:
direction_depth_summary = pd.crosstab(
    resolved_linkages["target_mode"],
    resolved_linkages["terminal_depth"],
    normalize="index",
)
direction_depth_summary.columns = [
    f"depth_{int(depth)}" for depth in direction_depth_summary.columns
]
resolved_linkages["seed_year"] = resolved_linkages["seed_time"].dt.year
year_depth_summary = resolved_linkages.groupby("seed_year", sort=True).agg(
    lineages=("seed_small_event_id", "size"),
    reached_medium_rate=("terminal_depth", lambda values: values.ge(1).mean()),
    reached_large_rate=("terminal_depth", lambda values: values.ge(2).mean()),
    reached_macro_rate=("terminal_depth", lambda values: values.eq(3).mean()),
)
display(direction_depth_summary.style.format("{:.2%}"))
display(year_depth_summary.style.format({
    "lineages": "{:,}",
    "reached_medium_rate": "{:.2%}",
    "reached_large_rate": "{:.2%}",
    "reached_macro_rate": "{:.2%}",
}))

,depth_0,depth_1,depth_2,depth_3
target_mode,,,,
down,56.35%,16.19%,7.99%,19.47%
up,53.11%,17.84%,8.30%,20.75%


,lineages,reached_medium_rate,reached_large_rate,reached_macro_rate
seed_year,,,,
2023,204,44.61%,27.45%,20.59%
2024,243,48.56%,31.28%,22.63%
2025,302,45.70%,27.48%,18.21%
2026,221,41.63%,26.70%,19.46%


## Four-depth price-path casebook

One recent strict-clean lineage is selected for each terminal depth. Selection and terminal markers use hindsight. Read each chart from the black small-seed marker forward before inspecting later hand-offs.

In [16]:
casebook_lineages = (
    strict_clean_linkages.sort_values("seed_time")
    .groupby("terminal_depth", sort=True, group_keys=False)
    .tail(1)
    .sort_values("terminal_depth")
)
display(casebook_lineages[[
    "seed_small_event_id", "seed_time", "target_mode",
    "medium_event_id", "medium_time", "large_event_id", "large_time",
    "terminal_event_id", "terminal_time", "terminal_depth",
    "small_medium_bars", "medium_large_bars", "large_macro_bars",
    "total_resolution_bars",
]])

,seed_small_event_id,seed_time,target_mode,medium_event_id,medium_time,large_event_id,large_time,terminal_event_id,terminal_time,terminal_depth,small_medium_bars,medium_large_bars,large_macro_bars,total_resolution_bars
1004,S2695,2026-11-06 17:20:00+02:00,up,<NA>,NaT,<NA>,NaT,S2696,2026-11-06 21:45:00+02:00,0,53,<NA>,<NA>,53.0
998,S2684,2026-10-30 15:45:00+02:00,down,M836,2026-10-30 17:00:00+02:00,<NA>,NaT,M837,2026-11-02 10:35:00+02:00,1,15,211,<NA>,226.0
990,S2655,2026-10-13 08:45:00+02:00,up,M829,2026-10-13 20:05:00+02:00,L387,2026-10-14 03:50:00+02:00,L388,2026-10-14 15:00:00+02:00,2,136,93,134,363.0
1003,S2694,2026-11-05 21:15:00+02:00,down,M838,2026-11-05 23:05:00+02:00,L392,2026-11-06 08:45:00+02:00,X220,2026-11-06 11:05:00+02:00,3,22,116,28,166.0


In [17]:
def plot_linkage_casebook(close: pd.Series, examples: pd.DataFrame) -> go.Figure:
    figure = make_subplots(
        rows=len(examples),
        cols=1,
        vertical_spacing=min(0.045, 0.16 / len(examples)),
        subplot_titles=[
            (
                f"depth {int(example.terminal_depth)} | target={example.target_mode} | "
                f"seed={example.seed_small_event_id}"
            )
            for example in examples.itertuples(index=False)
        ],
    )
    depth_colors = {0: "#64748b", 1: "#2563eb", 2: "#7c3aed", 3: "#c026d3"}

    for row_number, example in enumerate(examples.itertuples(index=False), start=1):
        window_start = max(0, int(example.seed_position) - CASEBOOK_BARS_BEFORE)
        window_end = min(
            len(close), int(example.terminal_position) + CASEBOOK_BARS_AFTER + 1
        )
        price_window = close.iloc[window_start:window_end]
        active_path = close.iloc[int(example.seed_position):int(example.terminal_position) + 1]
        depth = int(example.terminal_depth)

        figure.add_trace(go.Scattergl(
            x=price_window.index,
            y=price_window.to_numpy(),
            mode="lines",
            name="Close",
            legendgroup="close",
            showlegend=row_number == 1,
            line={"color": "#94a3b8", "width": 1.0},
            hovertemplate="time=%{x}<br>close=%{y:.5f}<extra></extra>",
        ), row=row_number, col=1)
        figure.add_trace(go.Scattergl(
            x=active_path.index,
            y=active_path.to_numpy(),
            mode="lines",
            name="Pinned lineage",
            legendgroup="lineage",
            showlegend=row_number == 1,
            line={"color": depth_colors[depth], "width": 3.0},
            hovertemplate="active lineage<br>time=%{x}<br>close=%{y:.5f}<extra></extra>",
        ), row=row_number, col=1)

        markers = [
            ("Small seed", example.seed_time, "#111827", "diamond"),
            ("Medium hand-off", example.medium_time, "#2563eb", "circle"),
            ("Large hand-off", example.large_time, "#7c3aed", "square"),
            ("Terminal resolution", example.terminal_time, depth_colors[depth], "x"),
        ]
        for label, marker_time, color, symbol in markers:
            if pd.isna(marker_time):
                continue
            marker_position = close.index.get_loc(marker_time)
            figure.add_trace(go.Scatter(
                x=[marker_time],
                y=[close.iloc[marker_position]],
                mode="markers",
                name=label,
                legendgroup=label,
                showlegend=row_number == 1,
                marker={"color": color, "size": 9, "symbol": symbol},
                hovertemplate=f"{label}<br>time=%{{x}}<br>price=%{{y:.5f}}<extra></extra>",
            ), row=row_number, col=1)
        figure.update_yaxes(title_text="Close", row=row_number, col=1)

    figure.update_layout(
        title=f"{SYMBOL}: exact four-scale lineage casebook",
        template="plotly_white",
        height=max(850, 330 * len(examples)),
        hovermode="closest",
        legend={"orientation": "h", "y": 1.05, "x": 0.0},
    )
    return figure


plot_linkage_casebook(prices, casebook_lineages).show()

# Four-threshold linkage completion gate

Before introducing live four-stage features, be able to explain:

1. Why a valid root requires broad agreement on the previous bar.
2. Why the initiating small event remains the lineage ID through every stage.
3. Why the medium and large event IDs are hand-off keys rather than replacement seeds.
4. Why exact event-ID equality is stronger than timestamp proximity.
5. What terminal depths 0, 1, 2, and 3 mean.
6. On which DCC each depth becomes attained and each stage outcome becomes known.
7. Why terminal depth remains hindsight until the lineage ends.
8. Why same-bar propagation has no unresolved live interval, and why excluding it biases the strict-clean ladder low.
9. Why ambiguous, unresolved, and unlinked paths remain explicit.
10. Why the linked probability product uses conditioning rather than independence.
11. Why linked stage rates differ from unconditional adjacent-edge rates here, but coincide at three scales.
12. How smaller-scale DCCs can occur inside unresolved later stages.
13. Why a depth-3 lineage need not be one monotonic price leg.
15. Why the product identity holds by arithmetic and is therefore not a test.
14. Why these frequencies describe structure rather than trading alpha.

The next step is four-threshold causal live development: retain the immutable small seed while measuring each unresolved stage using only information available at the observation bar.